In [1]:
# =========================
# Text-only baseline model
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# =========================
# 1. Load data
# =========================
# Update the paths if needed
X_TEXT_PATH = "X_text.csv"
Y_PATH = "deepseek_Y_quantile_final.csv"
BRIDGE_PATH = "Final_Matched_Universe_Full.csv"

x_text = pd.read_csv(X_TEXT_PATH)
y = pd.read_csv(Y_PATH)
bridge = pd.read_csv(BRIDGE_PATH)

print("x_text shape:", x_text.shape)
print("y shape:", y.shape)
print("bridge shape:", bridge.shape)

print("\nx_text columns:")
print(x_text.columns.tolist())

print("\ny columns:")
print(y.columns.tolist())

print("\nbridge columns:")
print(bridge.columns.tolist())


# =========================
# 2. Detect key columns
# =========================
# We try to detect the relevant columns automatically.
# If your actual column names are different, adjust them manually.

def find_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

xtext_companyid_col = find_col(x_text, ["companyid", "company_id"])
xtext_companyname_col = find_col(x_text, ["companyname", "company_name"])

y_ticker_col = find_col(y, ["ticker", "tic"])
y_label_col = find_col(y, ["label"])

bridge_companyid_col = find_col(bridge, ["companyid", "company_id"])
bridge_ticker_col = find_col(bridge, ["ticker", "tic"])
bridge_companyname_col = find_col(bridge, ["companyname", "company_name", "title"])

print("\nDetected columns:")
print("x_text companyid:", xtext_companyid_col)
print("x_text companyname:", xtext_companyname_col)
print("y ticker:", y_ticker_col)
print("y label:", y_label_col)
print("bridge companyid:", bridge_companyid_col)
print("bridge ticker:", bridge_ticker_col)
print("bridge companyname/title:", bridge_companyname_col)

if any(v is None for v in [xtext_companyid_col, y_ticker_col, y_label_col, bridge_companyid_col, bridge_ticker_col]):
    raise ValueError("Some key columns were not detected. Please check the printed column names and edit them manually.")


# =========================
# 3. Keep only the necessary Y columns
# =========================
# Important:
# Do NOT use CAR / MaxDrawdown / Volatility / returns as X features,
# because they were used to construct the label and would cause data leakage.

y_keep = y[[y_ticker_col, y_label_col]].drop_duplicates().copy()

# Convert labels to binary values
# Vulnerable = 1
# Resilient = 0
label_map = {
    "Vulnerable": 1,
    "Resilient": 0
}
y_keep["label_binary"] = y_keep[y_label_col].map(label_map)

# Standardize the ticker column name
y_keep = y_keep.rename(columns={y_ticker_col: "ticker"})

print("\nY label counts:")
print(y_keep[y_label_col].value_counts(dropna=False))
print(y_keep["label_binary"].value_counts(dropna=False))


# =========================
# 4. Build the bridge table
# =========================
# We only need companyid and ticker for the merge.
bridge_keep_cols = [bridge_companyid_col, bridge_ticker_col]
if bridge_companyname_col is not None:
    bridge_keep_cols.append(bridge_companyname_col)

bridge_keep = bridge[bridge_keep_cols].drop_duplicates().copy()
bridge_keep = bridge_keep.rename(columns={
    bridge_companyid_col: "companyid_bridge",
    bridge_ticker_col: "ticker"
})

print("\nBridge preview:")
print(bridge_keep.head())


# =========================
# 5. Merge X_text with the bridge table
# =========================
model_text = x_text.merge(
    bridge_keep,
    left_on=xtext_companyid_col,
    right_on="companyid_bridge",
    how="inner"
)

print("\nAfter X_text + bridge:", model_text.shape)


# =========================
# 6. Merge text features with Y
# =========================
model_text = model_text.merge(
    y_keep,
    on="ticker",
    how="inner"
)

print("\nMerged text-only dataset shape:", model_text.shape)
print("\nMerged label counts:")
print(model_text[y_label_col].value_counts(dropna=False))

print("\nDuplicate tickers in merged data:", model_text["ticker"].duplicated().sum())
print("\nMissing values (top 30):")
print(model_text.isna().sum().sort_values(ascending=False).head(30))


# =========================
# 7. Define text feature columns
# =========================
# We exclude ID columns and highly redundant columns for a cleaner baseline.
# This gives us a more interpretable first text-only model.

drop_text_cols = {
    # IDs
    xtext_companyid_col,
    xtext_companyname_col,
    "companyid_bridge",
    "ticker",
    y_label_col,
    "label_binary",

    # Company name / title columns
    "companyname",
    "company_name",
    "title",

    # Highly redundant FinBERT features
    "pct_positive",
    "pct_negative",
    "pct_neutral",
    "mean_p_positive",
    "mean_p_negative",
    "mean_p_neutral",
    "ai_keyword_count",
}

text_feature_cols = [c for c in model_text.columns if c not in drop_text_cols]

print("\nText feature columns:")
print(text_feature_cols)
print("\nNumber of text features:", len(text_feature_cols))

X = model_text[text_feature_cols].copy()
y_binary = model_text["label_binary"].copy()

print("\nX shape:", X.shape)
print("y shape:", y_binary.shape)


# =========================
# 8. Define evaluation function
# =========================
def evaluate_model(X, y, model_type="logit"):
    """
    Evaluate a model using 5-fold stratified cross-validation.
    Returns accuracy, F1, and ROC-AUC.
    """

    if model_type == "logit":
        # Logistic Regression: scaling is needed
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])
    elif model_type == "rf":
        # Random Forest: scaling is not required
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ])
    else:
        raise ValueError("model_type must be either 'logit' or 'rf'")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    result = {
        "model_type": model_type,
        "n_features": X.shape[1],
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }
    return result


# =========================
# 9. Run two baseline models
# =========================
results = []

results.append(evaluate_model(X, y_binary, model_type="logit"))
results.append(evaluate_model(X, y_binary, model_type="rf"))

results_df = pd.DataFrame(results)

print("\n=== Text-only baseline results ===")
print(results_df.round(4).to_string(index=False))


# =========================
# 10. Save outputs
# =========================
model_text.to_csv("model_text_only.csv", index=False)
results_df.to_csv("baseline_results_text_only.csv", index=False)

print("\nSaved files:")
print("  model_text_only.csv")
print("  baseline_results_text_only.csv")

x_text shape: (400, 28)
y shape: (158, 17)
bridge shape: (404, 6)

x_text columns:
['companyid', 'companyname', 'n_sentences', 'mean_p_positive', 'mean_p_negative', 'mean_p_neutral', 'mean_sentiment', 'std_sentiment', 'pct_negative', 'pct_neutral', 'pct_positive', 'ratio_strong_negative', 'ratio_strong_positive', 'topic_00_energy_utilities', 'topic_01_advertising_marketing', 'topic_02_ai_semiconductor_design', 'topic_03_ai_enterprise_healthcare_services', 'topic_04_financial_reporting_geographic', 'topic_05_financial_reporting_accounting', 'topic_06_government_defense', 'topic_07_project_execution_capital', 'topic_08_china_ai_semiconductor_supply_chain', 'topic_09_ai_cloud_infrastructure', 'topic_10_chinese_gaming_streaming', 'topic_11_ai_saas_cybersecurity', 'ai_topic_loading_sum', 'ai_keyword_count', 'ai_keyword_ratio']

y columns:
['company_id', 'cik', 'ticker', 'company_name', 'sic', 'sic_description', 'start_date_actual', 'start_price', 'end_date_actual', 'end_price', 'raw_return'